# Sustainable Development Index Computing

This trimmed notebook keeps only the indicator-system workflow: load the merged data, harmonize updated variables, screen indicators, compute CRITIC weights, and export the indicator-system outputs for reporting.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import variation
from scipy.stats.mstats import winsorize
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, minmax_scale
from pandas.api.types import is_numeric_dtype
from statsmodels.stats.outliers_influence import variance_inflation_factor

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
raw_route = Path.home() / 'OneDrive' / 'Rawdata'
config_route = Path.home() / 'OneDrive' / 'PhD Dissertation' / 'Data_Code' / 'Data'
years = list(range(2001, 2022))

input_path = project_dir / 'df_final.csv'
if not input_path.exists():
    input_path = raw_route / 'Data cleaning' / 'df_final.csv'

df_final = pd.read_csv(input_path, low_memory=False).set_index(['Numeric', 'Year']).sort_index()
df_final = df_final.loc[~df_final.index.duplicated(keep='first')].copy()
df_final.drop([col for col in df_final.columns if col.endswith('_y')], axis=1, inplace=True, errors='ignore')

coverage_all = (
    df_final.reset_index()[['Numeric', 'Year']]
    .dropna()
    .assign(Year=lambda x: pd.to_numeric(x['Year'], errors='coerce'))
    .dropna(subset=['Year'])
)
coverage_all['Year'] = coverage_all['Year'].astype(int)


In [2]:
alias_map = {
    'GDP per capita growth (annual %)_x': 'GDP per capita growth (annual %)',
    'Agriculture, forestry, and fishing, value added per worker (constant 2015 US$)_x': 'Agriculture, forestry, and fishing, value added per worker (constant 2015 US$)',
    'Industry (including construction), value added per worker (constant 2015 US$)_x': 'Industry (including construction), value added per worker (constant 2015 US$)',
    'Services, value added per worker (constant 2015 US$)_x': 'Services, value added per worker (constant 2015 US$)',
    'Current account balance, percent of GDP (Percent of GDP)(IMF)': 'Current account balance (% of GDP)',
    'Proportion of seats held by women in national parliaments (%)_x': 'Proportion of seats held by women in national parliaments (%)',
    'Prevalence of HIV, total (% of population ages 15-49)_x': 'Prevalence of HIV, total (% of population ages 15-49)',
    'Individuals using the Internet (% of population)_x': 'Individuals using the Internet (% of population)',
    'Access to electricity (% of population)_x': 'Access to electricity (% of population)',
    'Mortality caused by road traffic injury (per 100,000 population)_x': 'Mortality caused by road traffic injury (per 100,000 population)',
    'People using at least basic drinking water services (% of population)_x': 'People using at least basic drinking water services (% of population)',
    'Renewable internal freshwater resources per capita (cubic meters)_x': 'Renewable internal freshwater resources per capita (cubic meters)',
    'Energy intensity level of primary energy (MJ/$2017 PPP GDP)_x': 'Energy intensity level of primary energy (MJ/$2021 PPP GDP)',
    'Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal)_x': 'Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal)',
    'Forest area (% of land area)_x': 'Forest area (% of land area)',
    'CO2 emissions (metric tons per capita)_x': 'CO2 emissions (metric tons per capita)',
}
for target, source in alias_map.items():
    if target not in df_final.columns and source in df_final.columns:
        df_final[target] = df_final[source]

if 'Life expectancy at birth, total (years)_pct_change' not in df_final.columns:
    df_final['Life expectancy at birth, total (years)_pct_change'] = (
        df_final.groupby('Numeric')['Life expectancy at birth, total (years)'].pct_change()
    )
if 'Solar, tide, wave, fuel cell electricity installed capacity (million kilowatts)_per_capita' not in df_final.columns:
    df_final['Solar, tide, wave, fuel cell electricity installed capacity (million kilowatts)_per_capita'] = (
        df_final['Solar, tide, wave, fuel cell electricity installed capacity (million kilowatts)']
        / df_final['Population, total']
    ) * 10000
if 'Biomass and waste electricity net generation (million metric tons of oil equivalent)_per_capita' not in df_final.columns:
    df_final['Biomass and waste electricity net generation (million metric tons of oil equivalent)_per_capita'] = (
        df_final['Biomass and waste electricity net generation (million metric tons of oil equivalent)']
        / df_final['Population, total']
    ) * 10000
if 'Labor force(% of total population)' not in df_final.columns:
    df_final['Labor force(% of total population)'] = df_final['Labor force, total'] / df_final['Population, total']
if 'Merchandise exports (% of GDP)' not in df_final.columns:
    df_final['Merchandise exports (% of GDP)'] = df_final['Merchandise exports (current US$)'] / df_final['GDP (current US$)']
if 'Prevalence of undernourishment (percent) (3-year average)' not in df_final.columns:
    for source in ['Prevalence of undernourishment [2.1.1]', 'Prevalence of undernourishment (% of population)']:
        if source in df_final.columns:
            df_final['Prevalence of undernourishment (percent) (3-year average)'] = df_final[source]
            break
if 'CO2 emissions (metric tons per capita)' not in df_final.columns and 'CO2 emissions (million metric tonnes carbon dioxide)' in df_final.columns:
    df_final['CO2 emissions (metric tons per capita)'] = (
        df_final['CO2 emissions (million metric tonnes carbon dioxide)'] * 1_000_000
        / df_final['Population, total']
    )
if 'Scientific and technical journal articles per million people' not in df_final.columns and 'Scientific and technical journal articles' in df_final.columns:
    df_final['Scientific and technical journal articles per million people'] = (
        df_final['Scientific and technical journal articles'] * 1_000_000
        / df_final['Population, total']
    )
if 'CO2 emissions (metric tons per capita)_x' not in df_final.columns and 'CO2 emissions (metric tons per capita)' in df_final.columns:
    df_final['CO2 emissions (metric tons per capita)_x'] = df_final['CO2 emissions (metric tons per capita)']
if 'Terrestrial biome protection (global weights)' not in df_final.columns and 'Terrestrial Biome Protection (national weights)' in df_final.columns:
    df_final['Terrestrial biome protection (global weights)'] = df_final['Terrestrial Biome Protection (national weights)']
if 'PM2.5 exposure/Ambient particulate matter pollution' not in df_final.columns:
    for source in ['Anthropogenic PM2.5 exposure', 'PM2.5 exposure (PME)']:
        if source in df_final.columns:
            df_final['PM2.5 exposure/Ambient particulate matter pollution'] = df_final[source]
            break
if 'Net official development assistance and official aid received (current US$)(% of GNI)' not in df_final.columns and {'Net official development assistance and official aid received (current US$)', 'GNI (current US$)'}.issubset(df_final.columns):
    df_final['Net official development assistance and official aid received (current US$)(% of GNI)'] = (
        df_final['Net official development assistance and official aid received (current US$)']
        / df_final['GNI (current US$)']
    )

wetland_proxy_components = [
    'Shrubs and/or herbaceous vegetation, aquatic or regularly flooded|1000 HA|ECCCV|Shrubs and/or Herbaceous Vegetation and Aquatic Or Regularly Flooded|Environment, Climate Change, Climate and Weather, Land Cover Accounts, Shrubs and/or Herbaceous Vegetation and Aquatic Or Regularly Flooded|Climate regulating',
    'Mangroves|1000 HA|ECCCM|Mangroves|Environment, Climate Change, Climate and Weather, Land Cover Accounts, Mangroves|Climate regulating',
]
wetland_proxy_available = [c for c in wetland_proxy_components if c in df_final.columns]
if 'Wetland area(% of land area)' not in df_final.columns and wetland_proxy_available and 'Land area (sq. km)' in df_final.columns:
    wetland_area_sqkm = df_final[wetland_proxy_available].fillna(0).sum(axis=1) * 10
    df_final['Wetland area(% of land area)'] = (wetland_area_sqkm / df_final['Land area (sq. km)']) * 100

grassland_col = 'Grassland|1000 HA|ECCCG|Grassland|Environment, Climate Change, Climate and Weather, Land Cover Accounts, Grassland|Climate regulating'
if 'Grassland area(% of land area)' not in df_final.columns and grassland_col in df_final.columns:
    df_final['Grassland area(% of land area)'] = (
        df_final[grassland_col] * 10 / df_final['Land area (sq. km)']
    ) * 100

for canonical, legacy in [
    ('Wetland area(% of land area)', 'Wetland area（% of land area)'),
    ('Grassland area(% of land area)', 'Grassland area（% of land area)'),
]:
    if canonical not in df_final.columns and legacy in df_final.columns:
        df_final[canonical] = df_final[legacy]
    if legacy not in df_final.columns and canonical in df_final.columns:
        df_final[legacy] = df_final[canonical]

barren_col = 'Terrestrial barren land|1000 HA|ECCCT|Terrestrial Barren Land|Environment, Climate Change, Climate and Weather, Land Cover Accounts, Terrestrial Barren Land|Climate neutral'
old_barren_pct_col = 'Terrestrial barren land|1000 HA|ECCCT|Terrestrial Barren Land|Environment, Climate Change, Climate Indicators, Land Cover Accounts, Terrestrial Barren Land|Climate neutral(% of land area)'
if old_barren_pct_col not in df_final.columns and barren_col in df_final.columns:
    df_final[old_barren_pct_col] = (df_final[barren_col] * 10 / df_final['Land area (sq. km)']) * 100



In [3]:
VARIABLES = pd.read_excel(
    project_dir / 'Variables Chosen.xlsx',
    sheet_name='Index',
    na_values='..',
)
country = pd.read_excel(
    project_dir / 'Variables Chosen.xlsx',
    sheet_name='Countries',
    na_values='..',
)
dfgeo = pd.read_excel(
    raw_route / 'Country Classification' / 'UN Classification_Natural resources_Geography.xlsx',
    sheet_name='Sheet1',
    na_values='..',
    usecols='D, E',
).query("Region != 'N'")
country = country.merge(dfgeo, on=['Numeric'])

VARIABLES['Variables'] = VARIABLES['Variables'].replace({
    'Wetland area（% of land area)': 'Wetland area(% of land area)',
    'Grassland area（% of land area)': 'Grassland area(% of land area)',
})

screening_path = project_dir / 'country_sample_screening_2001_2021.csv'
excluded_alpha3 = {'MYT', 'REU', 'SSD', 'SOM'}
if screening_path.exists():
    screening = pd.read_csv(screening_path)
    keep_numeric = pd.to_numeric(
        screening.loc[screening['pass_all_rules'], 'Numeric'],
        errors='coerce',
    ).dropna().astype(int).unique()
    country['Numeric'] = pd.to_numeric(country['Numeric'], errors='coerce')
    country = country[country['Numeric'].isin(keep_numeric)].copy()
else:
    if 'Alpha-3 code' in country.columns:
        country = country[~country['Alpha-3 code'].isin(excluded_alpha3)].copy()
country['Numeric'] = country['Numeric'].astype(int)

VARIABLES.loc[
    VARIABLES['Variables'].eq('Wetland area(% of land area)'),
    '来源'
] = '联合国粮食及农业组织FAOSTAT土地覆盖数据库（经IMF-CID整理）'

missing_indicators = pd.DataFrame({
    'Variables': [
        v for v in VARIABLES.query("变量类型=='指标体系'")['Variables']
        if v not in set(df_final.columns)
    ]
})
missing_indicators.to_csv(project_dir / 'indicator_system_missing_indicators.csv', index=False, encoding='utf-8-sig')

VARIABLES = VARIABLES[~VARIABLES['Variables'].isin(missing_indicators['Variables'])].copy()

variables_available = (
    df_final.reset_index()
    .query('Year in @years')
    .groupby('Numeric')
    .count()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={0: 'count'})
)
variables_available['count'] = variables_available['count'] / len(years)
variables_available = variables_available.query('count > 0')
variables_available.to_excel(project_dir / 'Variables Count.xlsx', index=False)
VARIABLES[["一级指标", "二级指标", "三级指标", "来源"]].set_index([
    "一级指标", "二级指标", "三级指标"
]).to_excel(project_dir / 'Variables_name.xlsx', index=True)

coverage_summary = pd.DataFrame({
    'metric': ['all_merged_countries', 'all_merged_year_min', 'all_merged_year_max', 'configured_sample_countries'],
    'value': [
        coverage_all['Numeric'].nunique(),
        int(coverage_all['Year'].min()),
        int(coverage_all['Year'].max()),
        country['Numeric'].nunique(),
    ],
})
coverage_summary


,metric,value
0,all_merged_countries,57
1,all_merged_year_min,1750
2,all_merged_year_max,2050
3,configured_sample_countries,48


In [4]:
data_raw = df_final[VARIABLES.query("变量类型=='指标体系'")['Variables']].reset_index()
data_raw['Year'] = pd.to_numeric(data_raw['Year'], errors='coerce')
data_raw = data_raw.dropna(subset=['Numeric', 'Year'])
data_raw['Numeric'] = data_raw['Numeric'].astype(int)
data_raw['Year'] = data_raw['Year'].astype(int)
data_raw = data_raw.merge(country[['Numeric', 'Alpha-3 code', 'CountryName_CN']], on='Numeric')
data_filtered = data_raw.copy().query('Year in @years').set_index(['Numeric', 'Year'])

def interpolate_with_linear_regression(df, polynomial=False):
    df = df.copy()
    for col in df.columns:
        if df[col].isna().values.any():
            na_mask = df[col].isna()
            if (~na_mask).sum() < 10:
                continue
            if not is_numeric_dtype(df[col]):
                continue
            lin_reg = LinearRegression()
            not_na_years = df.loc[~na_mask].index.get_level_values(1).values.reshape((-1, 1))
            na_years = df.loc[na_mask].index.get_level_values(1).values.reshape((-1, 1))
            if polynomial:
                poly = PolynomialFeatures(2, include_bias=False)
                X = poly.fit_transform(not_na_years)
                lin_reg.fit(X, df.loc[~na_mask, col])
                pred = lin_reg.predict(poly.fit_transform(na_years))
            else:
                lin_reg.fit(not_na_years, df.loc[~na_mask, col])
                pred = lin_reg.predict(na_years)
            df.loc[na_mask, col] = pred
    return df


data_filled = data_filtered.groupby('Numeric').apply(interpolate_with_linear_regression).droplevel(0)


# {
#     'sample_countries_in_indicator_workflow': int(usable_country_year['Numeric'].nunique()),
#     'usable_year_min': int(usable_country_year['Year'].min()),
#     'usable_year_max': int(usable_country_year['Year'].max()),
#     'missing_indicator_count_before_cv_vif': int(len(missing_indicators)),
# }


In [5]:
meta_cols = ['Numeric', 'Year', 'Alpha-3 code', 'CountryName_CN', 'Region', 'incomegroup']
data_CV = data_filled.reset_index().set_index(['Alpha-3 code', 'Numeric', 'Year', 'CountryName_CN'])
coefva = pd.DataFrame(data_CV.apply(lambda s: abs(variation(s, ddof=1))), columns=['coefva'])
small_coefva = coefva.query('coefva < 0.25').index
variables_post_cv = [col for col in data_filled.columns if col not in small_coefva and col not in meta_cols]

(
    VARIABLES.query('类型 in ["正向", "负向"] and 变量类型 == "指标体系" and Variables in @variables_post_cv')
    .set_index(['一级指标', '二级指标', '三级指标'])
    .drop(columns=['变量类型'])
    .apply(lambda x: abs(variation(data_CV[x['Variables']], ddof=1)), axis=1)
    .to_frame(name='coefva')
    .round(2)
).to_excel(project_dir / 'Variables_CV.xlsx', index=True)

dropped_variables = []


def drop_according_VIF(variables):
    cols = list(set(data_CV.columns) & set(variables['Variables']))
    df = data_CV[cols].replace([np.inf, -np.inf], np.nan).dropna()
    vif = pd.DataFrame({'Variables': df.columns})
    vif['VIF'] = np.nan
    if df.shape[0] > 1 and df.shape[1] >= 2:
        vif['VIF'] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
        max_vif = vif.loc[vif['VIF'].idxmax()]
        if max_vif['VIF'] > 7.5:
            dropped_variables.append(max_vif)
            variables = drop_according_VIF(variables[variables['Variables'] != max_vif['Variables']])
    return variables

variables_post_VIF = list(
    VARIABLES.query('类型 in ["正向", "负向"] and 变量类型 == "指标体系" and Variables in @variables_post_cv')
    .groupby('一级指标', group_keys=False)
    .apply(drop_according_VIF)['Variables']
)
VARIABLES_POST_CV_VIF = VARIABLES.query("Variables in @variables_post_VIF and 变量类型 == '指标体系'").reset_index(drop=True)
VARIABLES_POST_CV_VIF[["一级指标", "二级指标", "三级指标", "类型", "来源"]].set_index([
    "一级指标", "二级指标", "三级指标"
]).to_excel(project_dir / 'Variables_vif2.xlsx', index=True)

selected_variables = VARIABLES_POST_CV_VIF[["一级指标", "二级指标", "三级指标", "Variables", "类型", "来源"]]
selected_variables.to_excel(project_dir / 'selected_indicator_system_variables.xlsx', index=False)
selected_variables


,一级指标,二级指标,三级指标,Variables,类型,来源
0,经济,发展质量,人均GDP增长率,GDP per capita growth (annual %)_x,正向,世界银行WDI数据库
1,经济,发展质量,商品出口占GDP比重,Merchandise exports (% of GDP),正向,世界银行WDI数据库
2,经济,发展质量,最终消费支出占GDP百分比,Final consumption expenditure (% of GDP),正向,世界银行WDI数据库
3,经济,发展质量,通货膨胀率,"Inflation, GDP deflator (annual %)",负向,世界银行WDI数据库
4,经济,产业发展,每工人农业增加值,"Agriculture, forestry, and fishing, value adde...",正向,世界银行WDI数据库
5,经济,产业发展,每工人工业增加值,"Industry (including construction), value added...",正向,世界银行WDI数据库
6,经济,产业发展,每工人服务业增加值,"Services, value added per worker (constant 201...",正向,世界银行WDI数据库
7,经济,外部经济联系,经常账户余额占GDP百分比,"Current account balance, percent of GDP (Perce...",正向,世界银行WDI数据库
8,社会,社会公平,性别平等,Proportion of seats held by women in national ...,正向,世界银行WDI数据库
9,社会,社会公平,营养不良发生率,Prevalence of undernourishment (percent) (3-ye...,负向,联合国粮食及农业组织FAO数据库


In [6]:
data1 = (
    data_raw.copy()
    .query('Year >= @years[0] and Year <= @years[-1]')
    .set_index(['Numeric', 'Year', 'Alpha-3 code', 'CountryName_CN'])[VARIABLES_POST_CV_VIF['Variables']]
    .reset_index()
)
data1.to_excel(project_dir / 'data1.xlsx', index=False)
data1 = data1.set_index(['Numeric', 'Year'])

data_filled = data1.groupby('Numeric').apply(interpolate_with_linear_regression).droplevel(0)
data_filled['Inflation, GDP deflator (annual %)'] = winsorize(
    data_filled['Inflation, GDP deflator (annual %)'], limits=[0.01, 0.01]
)
data_filled['Inflation, GDP deflator (annual %)'] = data_filled['Inflation, GDP deflator (annual %)'].abs()

selected_index_vars = VARIABLES_POST_CV_VIF['Variables'].to_list()
indicator_coverage = data_filled[selected_index_vars].notna().sum(axis=1).rename('available_indicators').to_frame()
indicator_coverage['total_indicators'] = len(selected_index_vars)
indicator_coverage['coverage_rate'] = indicator_coverage['available_indicators'] / indicator_coverage['total_indicators']
indicator_coverage['meets_80pct_coverage'] = indicator_coverage['coverage_rate'] >= 0.8
indicator_coverage_export = indicator_coverage.reset_index().merge(
    data_filled.reset_index()[['Numeric', 'Year', 'Alpha-3 code', 'CountryName_CN']],
    on=['Numeric', 'Year'],
    how='left',
)
indicator_coverage_export.to_csv(
    project_dir / 'indicator_country_year_coverage_80pct.csv',
    index=False,
    encoding='utf-8-sig',
)

fea_posi = data_filled[
    np.intersect1d(data_filled.columns, VARIABLES_POST_CV_VIF.query('类型 == "正向"')['Variables'])
].copy()
fea_nega = data_filled[
    np.intersect1d(data_filled.columns, VARIABLES_POST_CV_VIF.query('类型 == "负向"')['Variables'])
].copy()
fea_posi.loc[:] = minmax_scale(fea_posi)
fea_nega.loc[:] = minmax_scale(-fea_nega)
scaled_data = (
    pd.merge(fea_posi, fea_nega, how='outer', left_index=True, right_index=True)
    .query('Year >= @years[0]')
    .reset_index()
    .merge(country[['Numeric', 'CountryName_CN']], on='Numeric')
    .set_index(['CountryName_CN', 'Numeric', 'Year'])
)
scaled_data.to_excel(project_dir / 'scaled_data.xlsx', index=True)

# CRITIC weights are estimated on available standardized values. For index construction,
# remaining missing standardized values are assigned zero contribution, and country-years
# with less than 80% post-imputation indicator coverage are excluded from comparison.
coverage_key = pd.MultiIndex.from_frame(scaled_data.reset_index()[['Numeric', 'Year']])
valid_country_year = indicator_coverage['meets_80pct_coverage'].reindex(coverage_key).to_numpy(dtype=bool)
scaled_data_for_index = scaled_data.fillna(0)

criticweight = ((1 - scaled_data.corr().abs()).sum() * scaled_data.std())
criticweight = criticweight / criticweight.sum()
critic_weight_export = criticweight.rename('weight').to_frame().merge(
    VARIABLES_POST_CV_VIF[['一级指标', '二级指标', '三级指标', 'Variables', '类型']],
    left_index=True,
    right_on='Variables',
    how='left',
)
critic_weight_export.to_excel(project_dir / 'critic_weight.xlsx', index=False)
critic_weight_export.to_csv(project_dir / 'critic_weight.csv', index=False, encoding='utf-8-sig')

variables_by_level1 = {
    level1: VARIABLES_POST_CV_VIF.query(f'一级指标 == "{level1}"')['Variables'].to_list()
    for level1 in '经济 社会 资源 生态'.split()
}
level1_cn2en = {'经济': 'Economy', '社会': 'Society', '资源': 'Resource', '生态': 'Ecology'}
index_equal = pd.concat(
    [
        scaled_data_for_index[variables_by_level1[level1]].mean(axis=1).rename('SDI_' + level1_cn2en[level1] + '_Equal')
        for level1 in variables_by_level1
    ],
    axis=1,
)
index_equal['SDI_Equal'] = index_equal.mean(axis=1)
index_equal['SDI_Average'] = scaled_data_for_index[
    [v for variables in variables_by_level1.values() for v in variables]
].mean(axis=1)
index_data = pd.concat(
    [
        (
            (scaled_data_for_index * criticweight[variables_by_level1[level1]]).sum(axis=1)
            / criticweight[variables_by_level1[level1]].sum()
        ).rename('SDI_' + level1_cn2en[level1])
        for level1 in variables_by_level1
    ],
    axis=1,
)
index_data['SDI'] = (scaled_data_for_index * criticweight).sum(axis=1)

# ------------------------------------------------------------------
# Two-stage CRITIC: compute CRITIC weights separately within each
# dimension, then combine with equal dimension weights (0.25 each)
# ------------------------------------------------------------------
two_stage_dim_scores = {}
two_stage_dim_weights = {}

for level1, vars_in_dim in variables_by_level1.items():
    sub = scaled_data[vars_in_dim]  # use original (with NaN) for weight estimation
    crit_dim = ((1 - sub.corr().abs()).sum() * sub.std())
    crit_dim = crit_dim / crit_dim.sum()
    two_stage_dim_weights[level1] = crit_dim
    # score: missing values filled with 0 for aggregation
    two_stage_dim_scores[level1] = (scaled_data_for_index[vars_in_dim] * crit_dim).sum(axis=1)

for level1, en in level1_cn2en.items():
    index_data['SDI_' + en + '_TwoStage'] = two_stage_dim_scores[level1]

index_data['SDI_TwoStage'] = 0.25 * (
    index_data['SDI_Economy_TwoStage'] +
    index_data['SDI_Society_TwoStage'] +
    index_data['SDI_Resource_TwoStage'] +
    index_data['SDI_Ecology_TwoStage']
)
index_data = index_data.merge(index_equal, left_index=True, right_index=True)
index_data.loc[~valid_country_year, :] = np.nan

index_data_filled0 = index_data.reset_index().merge(
    data_filled.reset_index(),
    how='outer',
    on=['Year', 'Numeric', 'CountryName_CN'],
)
index_data_filled = index_data_filled0.merge(dfgeo, how='left', on='Numeric').set_index(
    ['Alpha-3 code', 'CountryName_CN', 'Numeric', 'Year']
)
index_data_filled.to_excel(project_dir / 'index_data.xlsx', index=True)
index_data_filled.to_csv(project_dir / 'index_data.csv', index=True, encoding='utf-8-sig')

summary = pd.DataFrame([
    {'metric': 'all_merged_countries', 'value': int(coverage_all['Numeric'].nunique())},
    {'metric': 'all_merged_year_min', 'value': int(coverage_all['Year'].min())},
    {'metric': 'all_merged_year_max', 'value': int(coverage_all['Year'].max())},
    {'metric': 'configured_sample_countries', 'value': int(country['Numeric'].nunique())},
    {'metric': 'indicator_workflow_countries_2001_2021', 'value': int(data_filtered.reset_index()['Numeric'].nunique())},
    {'metric': 'indicator_workflow_year_min', 'value': int(data_filtered.reset_index()['Year'].min())},
    {'metric': 'indicator_workflow_year_max', 'value': int(data_filtered.reset_index()['Year'].max())},
    {'metric': 'missing_indicator_count_before_cv_vif', 'value': int(len(missing_indicators))},
    {'metric': 'final_indicator_count_after_cv_vif', 'value': int(len(VARIABLES_POST_CV_VIF))},
    {'metric': 'minimum_country_year_indicator_coverage_rate_after_imputation', 'value': float(indicator_coverage['coverage_rate'].min())},
    {'metric': 'country_years_below_80pct_indicator_coverage', 'value': int((~indicator_coverage['meets_80pct_coverage']).sum())},
])
summary.to_csv(project_dir / 'indicator_system_summary.csv', index=False, encoding='utf-8-sig')
summary.to_excel(project_dir / 'indicator_system_summary.xlsx', index=False)
summary


,metric,value
0,all_merged_countries,57.000000
1,all_merged_year_min,1750.000000
2,all_merged_year_max,2050.000000
3,configured_sample_countries,48.000000
4,indicator_workflow_countries_2001_2021,45.000000
5,indicator_workflow_year_min,2001.000000
6,indicator_workflow_year_max,2021.000000
7,missing_indicator_count_before_cv_vif,0.000000
8,final_indicator_count_after_cv_vif,37.000000
9,minimum_country_year_indicator_coverage_rate_a...,0.810811
